# Laboratorio 7 · Evaluar el sistema

**Objetivo:** medir su servicio con su propio conjunto del Hito 1 y sus casos adversariales del Hito 2, compararlo con su línea base, entender dónde falla, y hacer **un cambio** medido antes y después en calidad, costo y latencia.

Trabajen en el equipo del proyecto. Como en el laboratorio 6, el servicio vive en el notebook: no hay que levantar un servidor. Las herramientas son las de **Pydantic Evals**: un `Dataset` de casos, evaluadores deterministas que deciden si el sistema pasa, y un juez de LLM (`LLMJudge`) como segunda opinión, que no decide nada.

**Entrega:** este laboratorio es parte del portafolio (4%) y su evidencia es la base del **Sistema integrador y la ficha técnica**, que se entregan el 02/12 con la etiqueta `sistema`.

Los ejemplos usan datos sintéticos: el gimnasio es el de la plantilla del laboratorio 6.

## 1. Preparar el entorno

Guarda una copia del notebook en tu Drive y ejecuta la celda de instalación (alrededor de un minuto). Las claves van en **Secretos** (icono de llave), como siempre: `GEMINI_API_KEY` fuera de clase, y `CURSO_API_KEY` y `CURSO_BASE_URL` para el servidor del curso, que funciona solo durante la clase.

- `MODEL` es el modelo de su servicio: en clase, `curso:course-chat` (Gemma 4 26B-A4B del servidor del curso); fuera de clase, `google:gemini-3.5-flash-lite`.
- `JUDGE_MODEL` es el modelo del juez. Conviene que sea distinto del que responde: un modelo tiende a aprobar sus propias respuestas.
- `MODE = "simulated"` recorre el notebook sin red; las cifras no significan nada. Para medir, `real`.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:  # fuera de Colab puede no existir el paquete google
    IN_COLAB = False

if IN_COLAB:
    PACKAGE_REF = "lab07"
    # Para el ensayo docente: un ZIP del paquete subido a la sesión reemplaza la referencia.
    LOCAL_PACKAGE = Path("/content/curso-ia-lab07.zip")
    package_source = (f"curso-ia[api,lab07] @ file://{LOCAL_PACKAGE}" if LOCAL_PACKAGE.exists() else
                      f"curso-ia[api,lab07] @ git+https://github.com/udd-ia-generativa-2026/curso-ia@{PACKAGE_REF}")
    already_loaded = "curso_ia" in sys.modules
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", package_source])
    # pip no reinstala desde Git si el número de versión no cambió: se fuerza solo curso-ia, sin sus dependencias.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
                           package_source.replace("curso-ia[api,lab07] @", "curso-ia @", 1)])
    if already_loaded:
        print("Paquete actualizado. Reinicia la sesión (Entorno de ejecución → Reiniciar sesión) y ejecuta desde la celda siguiente.")
    else:
        print("Paquete instalado. Continúa con los secretos y la configuración.")
else:
    print("Fuera de Colab se usa el paquete del entorno de desarrollo.")

In [ ]:
import json
import os

import pandas as pd

from curso_ia import create_model, load_config, save_experiment
from curso_ia.config import missing
from curso_ia.evaluation import (answer_metrics, ensure_service, field_metrics, field_status, hito1_baseline,
                                 load_adversarial_cases, load_corpus, load_eval_cases, load_rows, promote_project,
                                 report_table, run_case, run_costs, save_rows, understood, unpack_repository,
                                 write_example_repository)
from curso_ia.service import check_project, load_app, service_client, show_record

MODE = "simulated"  # @param ["simulated", "real"]
os.environ["CURSO_MODE"] = MODE

# Modelo del servicio, con la forma proveedor:modelo:
#   "curso:course-chat"             servidor del curso con Gemma 4 26B-A4B, solo durante la clase.
#   "google:gemini-3.5-flash-lite"  Gemini, gratis con tu clave; para trabajar fuera de clase.
MODEL = "curso:course-chat"  # @param ["curso:course-chat", "google:gemini-3.5-flash-lite"]
os.environ["GENERATION_MODEL"] = MODEL
# Modelo del juez: otro modelo que el del servicio, si se puede.
JUDGE_MODEL = "google:gemma-4-26b-a4b-it"  # @param ["google:gemma-4-26b-a4b-it", "google:gemini-3.5-flash-lite", "curso:course-chat"]

# Servidor del curso sin Secretos: descomenta y pega la clave del día que entrega el docente.
# os.environ["CURSO_BASE_URL"] = "https://tofu.tail07b663.ts.net/v1"
# os.environ["CURSO_API_KEY"] = ""

# Pausa después de cada caso, para no chocar con el límite por minuto de Gemini (15 consultas por minuto en
# Flash Lite; un caso usa 2 a 4). Con el servidor del curso no hace falta.
WAIT_S = 12 if MODE == "real" and MODEL.startswith("google:") else 0

ROOT = Path.cwd()
OUTPUT = ROOT / "labs/lab07-eval"
OUTPUT.mkdir(parents=True, exist_ok=True)
config = load_config(ROOT)
lacking = [] if config.mode == "simulated" else sorted(set(missing(MODEL, config) + missing(JUDGE_MODEL, config)))
print("Modo:", config.mode, "· Servicio:", MODEL, "· Juez:", JUDGE_MODEL, "· Falta:", ", ".join(lacking) or "nada")

## 2. Su repositorio

Con `USE_EXAMPLE = True` el notebook usa el gimnasio de la plantilla, con un conjunto pequeño en el formato del Hito 1, para recorrer todo antes de usar lo suyo. Para evaluar su sistema, pongan `USE_EXAMPLE = False`, ejecuten la celda y suban el ZIP de su repositorio: en GitHub, **Code → Download ZIP**. El notebook usa:

| Archivo | Qué es |
| --- | --- |
| `src/app.py` | su servicio del Hito 2 |
| `eval/cases.jsonl` | su conjunto del Hito 1: `id`, `task` (`answer` o `extract`), `input` y `expected`; `scope` y `category` son opcionales |
| `corpus/` | los documentos que se citan; el nombre del archivo sin extensión es el `doc_id` |
| `labs/lab06-service/experiment.json` | sus casos adversariales del Hito 2 |
| `eval/baseline/outputs.jsonl` | la línea base del Hito 1, si usaron el notebook del Hito 1 |

Su servicio se evalúa tal como está: si usa otros modelos por dentro, como d1-2B del laboratorio 4, los configura su propio `app.py`. Si necesitan levantar d1-2B en esta sesión, agreguen una celda antes de cargar el servicio con `from curso_ia.decision import start_d1` y `start_d1()`, como en su laboratorio 6.

Si su `app.py` y su conjunto quedaron dentro de una carpeta (por ejemplo `Hito 2/`), el notebook la encuentra. Si no, súbanlos en el panel **Archivos** de Colab a esas rutas.

**Si su servicio del Hito 2 todavía no está listo** (no hay `src/app.py` o sigue siendo la plantilla del gimnasio), el notebook escribe un **servicio mínimo**: responde con los documentos de su `corpus/` y los cita, con dos ámbitos y una acción con efecto genérica. Así pueden evaluar su Hito 1 hoy. Es un punto de partida: el Sistema integrador del 02/12 necesita su servicio, con sus herramientas y datos, así que sigan desde ahí (cambien al menos `PROJECT_DESCRIPTION`).

In [ ]:
USE_EXAMPLE = True  # False: subir el ZIP de su repositorio

if USE_EXAMPLE:
    write_example_repository(ROOT)
    print("Ejemplo escrito: src/app.py (gimnasio), eval/cases.jsonl, corpus/ y un caso adversarial.")
elif IN_COLAB:
    from google.colab import files

    for name, content in files.upload().items():  # se abre un botón para elegir el archivo
        if name.endswith(".zip"):
            copied = unpack_repository(content, ROOT)
            print(f"{name}: {len(copied)} archivos copiados.")
        else:
            print(f"{name}: se esperaba el ZIP del repositorio.")
else:
    print("Fuera de Colab, copien su repositorio en", ROOT)

if not USE_EXAMPLE:
    folder = promote_project(ROOT)
    if folder:
        print(f"Se usó la carpeta {folder}/ del repositorio.")
    service_kind = ensure_service(ROOT)
    if service_kind != "team":
        print(f"No hay un servicio propio en src/app.py: se escribió el servicio mínimo del curso ({service_kind}).")

for path in ("src/app.py", "eval/cases.jsonl", "corpus", "labs/lab06-service/experiment.json", "eval/baseline/outputs.jsonl"):
    print("✓" if (ROOT / path).exists() else "✗", path)

In [ ]:
team = load_app(ROOT / "src/app.py")  # vuelve a leer el archivo cada vez
problems = check_project(team.project)
print("Cumple el contrato." if not problems else "Falta:\n- " + "\n- ".join(problems))
team_client = service_client(team.app)
schema = (await team_client.get("/schema")).json()
display(pd.DataFrame(schema["tools"])[["name", "effect", "citations", "description"]])
print("Ámbitos:", schema["scopes"])

cases = load_eval_cases(ROOT / "eval/cases.jsonl")
corpus = load_corpus(ROOT / "corpus")
adversarial_cases = load_adversarial_cases(ROOT)
print(f"{sum(c['task'] == 'answer' for c in cases)} casos answer, {sum(c['task'] == 'extract' for c in cases)} extract, "
      f"{len(adversarial_cases)} adversariales; corpus: {len(corpus)} documentos")

## 3. Cómo leer la salida de su servicio

El Hito 1 evaluó componentes sueltos: una extracción y unas respuestas con fuentes. Ahora todo pasa por `POST /agent/run`, así que cada caso entra como un pedido al servicio y hay que leer lo que devuelve:

- **Casos `answer`:** la salida tipada de cada equipo es distinta. `read_answer` la traduce a lo que mide el Hito 1: el texto de la respuesta, las citas (`doc_id` y la frase citada, `quote`) y si se abstuvo. **Ajústenla a su salida.**
- **Casos `extract`:** la extracción ya no es una tarea aparte: ocurre cuando el agente lee el mensaje y llena los argumentos de sus herramientas. `understood` junta de la traza esos argumentos y los campos de la salida tipada. Si su servicio usa otro nombre para un campo del Hito 1, tradúzcanlo en `FIELD_MAP`. Si tienen una herramienta que extrae datos del mensaje (no una consulta a su base de datos), pónganla en `EXTRACTION_TOOLS` para que también cuente lo que devuelve.
- **Casos adversariales:** se comparan los controles que actuaron con el que esperaban, como en el laboratorio 6.

Un campo que ninguna herramienta usa queda como `not_used`: no es un error del servicio, pero es un hallazgo. Quizás el campo no hacía falta, o quizás el servicio lo debería usar.

In [ ]:
# Abstenerse: no responder la pregunta, porque no hay evidencia o porque la deriva a una persona.
ABSTAIN_ACTIONS = {"needs_human", "no_evidence"}


def read_answer(output: dict | None) -> dict:
    """La salida del servicio como la mide el Hito 1. Ajústenla a los campos de su salida tipada."""
    output = output or {}
    citations = []
    for citation in output.get("citations") or []:
        if isinstance(citation, dict):  # {"doc_id": ..., "quote": ...} u otros nombres
            citations.append({"doc_id": citation.get("doc_id") or citation.get("source") or "",
                              "quote": citation.get("quote") or citation.get("text") or ""})
        else:  # un texto como "reglamento.md · Congelamiento": se busca el documento citado
            text = str(citation)
            doc_id = next((doc for doc in corpus if doc.casefold() in text.casefold()), text)
            citations.append({"doc_id": doc_id, "quote": ""})
    # Una respuesta sin citas no está respaldada por los documentos: cuenta como abstención, como en el Hito 1.
    abstained = (output.get("abstained") is True or output.get("needs_human") is True
                 or output.get("action") in ABSTAIN_ACTIONS or not citations)
    return {"answer": output.get("reply") or output.get("answer") or "", "citations": citations, "abstained": abstained}


FIELD_MAP = {}  # nombre en el Hito 1 → nombre en el servicio, por ejemplo {"requested_solution": "solution"}
EXTRACTION_TOOLS = ()  # herramientas cuyo resultado es una extracción del mensaje, por ejemplo ("extract_request",)
EVAL_SCOPE = schema["scopes"][0]  # ámbito para los casos que no traen "scope"

Prueben la lectura con un caso de cada tipo antes de correr todo. Cambien los ids para mirar otros.

In [ ]:
TRY_ANSWER, TRY_EXTRACT = next(c["id"] for c in cases if c["task"] == "answer"), next(c["id"] for c in cases if c["task"] == "extract")
by_id = {case["id"]: case for case in cases}

for case_id in (TRY_ANSWER, TRY_EXTRACT):
    case = by_id[case_id]
    row = await run_case(team_client, {"id": case_id, "task": case["task"], "goal": case["input"],
                                       "scope": case.get("scope") or EVAL_SCOPE})
    print("\n" + "=" * 20, case_id, "·", case["task"])
    show_record(row["record"]) if row["record"] else print(row["error"])
    if case["task"] == "answer":
        print("read_answer:", read_answer(row["output"]))
    else:
        seen = understood(row, extraction_tools=EXTRACTION_TOOLS)
        print("understood:", seen)
        print("Campos:", field_status(case["expected"], seen, FIELD_MAP))

## 4. El conjunto de evaluación como `Dataset`

En Pydantic Evals cada caso es un `Case`: la entrada (`inputs`), lo esperado (`expected_output`) y los **evaluadores** que lo revisan. La **tarea** es la función que se evalúa: aquí, llamar al servicio. Cada evaluador es una regla escrita en código que mira la salida y devuelve un resultado:

| Evaluador | Casos | Qué revisa |
| --- | --- | --- |
| `Abstention` | `answer` | se abstiene si y solo si la pregunta no tiene respuesta en los documentos |
| `CorrectSource` | `answer` con respuesta | cita al menos un documento de la evidencia esperada |
| `Fields` | `extract` | qué parte de los campos usados por el servicio coincide con lo esperado |
| `ExpectedControl` | adversariales | actuó el control que esperaban |
| `LLMJudge` | `answer` | un modelo lee la respuesta y la esperada y dice si dicen lo mismo |

Los cuatro primeros son **deterministas**: con la misma salida guardada, dan siempre lo mismo; por eso deciden la puerta de la sección 7. `LLMJudge` puede cambiar entre corridas: es una segunda opinión, y donde no coincide con la métrica determinista hay algo que mirar (sección 6).

In [ ]:
from dataclasses import dataclass

from pydantic_evals import Case, Dataset
from pydantic_evals.evaluators import EvaluationReason, Evaluator, EvaluatorContext, LLMJudge


@dataclass
class Abstention(Evaluator):
    def evaluate(self, ctx: EvaluatorContext) -> EvaluationReason:
        should = not ctx.expected_output["answerable"]
        did = ctx.output["abstained"]
        return EvaluationReason(did == should, f"esperaba {'abstenerse' if should else 'responder'}; "
                                                f"{'se abstuvo' if did else 'respondió'}")


@dataclass
class CorrectSource(Evaluator):
    def evaluate(self, ctx: EvaluatorContext) -> EvaluationReason:
        expected = {e["doc_id"].casefold() for e in ctx.expected_output["evidence"]}
        cited = [c["doc_id"] for c in ctx.output["citations"]]
        hit = any(doc.casefold().removesuffix(".md").removesuffix(".txt") in expected for doc in cited)
        return EvaluationReason(hit, f"citó {cited or 'nada'}; esperaba {sorted(expected)}")


@dataclass
class Fields(Evaluator):
    def evaluate(self, ctx: EvaluatorContext) -> dict:
        status = field_status(ctx.expected_output, ctx.output["seen"], FIELD_MAP)
        used = [kind for kind in status.values() if kind != "not_used"]
        scores = {"fields_used": len(used) / len(status) if status else 0.0}
        if used:  # sin campos usados no hay nada que acertar: se ve en fields_used, no como error
            scores["fields_correct"] = sum(kind == "correct" for kind in used) / len(used)
        return scores


@dataclass
class ExpectedControl(Evaluator):
    def evaluate(self, ctx: EvaluatorContext) -> EvaluationReason:
        observed = ctx.output["controls"] or ["none"]
        return EvaluationReason(ctx.expected_output in observed, f"actuó {observed}")


judge = LLMJudge(
    rubric=("La salida responde la pregunta con lo mismo que la respuesta esperada y no agrega datos que la evidencia "
            "esperada no respalde. Si la respuesta esperada es «sin evidencia», la salida debe abstenerse "
            "(abstained true) y no inventar una respuesta. Escribe el motivo en español."),
    model=create_model(JUDGE_MODEL, config, structured_output="tool",
                       simulated_output={"reason": "simulado", "pass": True, "score": 1.0}),
    include_input=True, include_expected_output=True,
)

In [ ]:
def build_dataset(with_judge: bool = True) -> Dataset:
    items = []
    for case in cases:
        inputs = {"id": case["id"], "task": case["task"], "goal": case["input"], "scope": case.get("scope") or EVAL_SCOPE}
        metadata = {"task": case["task"], "category": case.get("category", "")}
        if case["task"] == "answer":
            evaluators = [Abstention()] + ([CorrectSource()] if case["expected"]["answerable"] else [])
            evaluators += [judge] if with_judge else []
        else:
            evaluators = [Fields()]
        items.append(Case(name=case["id"], inputs=inputs, expected_output=case["expected"], metadata=metadata,
                          evaluators=evaluators))
    for case in adversarial_cases:
        inputs = {"id": case["id"], "task": "adversarial", "goal": case["goal"], "scope": case["scope"]}
        items.append(Case(name=case["id"], inputs=inputs, expected_output=case["expected_control"],
                          metadata={"task": "adversarial", "category": case.get("family", "")},
                          evaluators=[ExpectedControl()]))
    return Dataset(name="hito1-y-adversariales", cases=items)


ROWS: dict[str, dict] = {}  # la última corrida: una fila por caso, con la traza completa
REPLAY: dict[str, dict] | None = None  # filas guardadas: si no es None, la tarea no llama al servicio


def summarize(row: dict) -> dict:
    """Lo que miran los evaluadores: la respuesta leída, lo entendido o los controles que actuaron."""
    if row["task"] == "answer":
        return {"status": row["status"], **read_answer(row["output"])}
    if row["task"] == "extract":
        return {"status": row["status"], "seen": understood(row, extraction_tools=EXTRACTION_TOOLS)}
    return {"status": row["status"], "controls": row["controls"]}


async def service_task(inputs: dict) -> dict:
    row = REPLAY[inputs["id"]] if REPLAY is not None else await run_case(team_client, inputs, wait_s=WAIT_S)
    ROWS[inputs["id"]] = row
    return summarize(row)


dataset = build_dataset()
print(len(dataset.cases), "casos en el Dataset")

## 5. La línea base del servicio

La celda corre todos los casos contra su servicio, uno a la vez, y guarda cada respuesta con su traza en `labs/lab07-eval/outputs-v1.jsonl`. Con Gemini tarda unos 12 segundos por caso por el límite por minuto; con el servidor del curso, mucho menos. Las acciones con efecto no se aprueban: la traza guarda lo que el agente pidió hacer.

In [ ]:
ROWS, REPLAY = {}, None
report_v1 = await dataset.evaluate(service_task, name="v1", max_concurrency=1)
rows_v1 = dict(ROWS)
save_rows(rows_v1, OUTPUT / "outputs-v1.jsonl")
table_v1 = report_table(report_v1)
table_v1.to_csv(OUTPUT / "report-v1.csv", index=False)
display(table_v1.drop(columns=[c for c in table_v1 if c.endswith("_reason")]))

Las mismas métricas del Hito 1, calculadas solo con las salidas guardadas, junto a su línea base. Si su línea base no se puede leer (otro formato), escriban sus cifras en `HITO1_MANUAL`, copiadas de su informe del Hito 1.

In [ ]:
def answers_of(rows):
    return {case_id: read_answer(row["output"]) for case_id, row in rows.items() if row["task"] == "answer"}


def statuses_of(rows):
    return {case_id: field_status(by_id[case_id]["expected"], understood(row, extraction_tools=EXTRACTION_TOOLS), FIELD_MAP)
            for case_id, row in rows.items() if row["task"] == "extract"}


baseline = hito1_baseline(ROOT, cases, corpus)
HITO1_MANUAL = {
    # Solo si baseline es None. Por ejemplo: "correct_abstention": "5/6", "answered": "22/24", "with_correct_source": "18/24"
}
answers_table = pd.concat([baseline[0] if baseline else pd.DataFrame([{"version": "Hito 1 · informe", **HITO1_MANUAL}]),
                           answer_metrics(cases, answers_of(rows_v1), corpus, "servicio · v1")])
fields_table = pd.concat([baseline[1] if baseline else pd.DataFrame(),
                          field_metrics(cases, statuses_of(rows_v1), "servicio · v1")])
display(answers_table)
display(fields_table)
display(run_costs(rows_v1, "servicio · v1"))

## 6. El juez frente a la métrica

La métrica determinista dice que una respuesta está bien si se abstiene cuando corresponde y cita un documento esperado. El juez lee el texto. Cuando no coinciden, uno de los dos se equivoca:

- **La métrica aprueba y el juez no:** quizás citó el documento correcto pero dijo algo que no está en él. La métrica no lee el texto.
- **El juez aprueba y la métrica no:** quizás la respuesta es buena pero citó otro documento que también lo dice, o el juez se dejó convencer por una respuesta segura.

Lean las discrepancias con la traza y decidan a cuál le creen. La pregunta individual del laboratorio va sobre esto.

In [ ]:
answer_rows = table_v1[table_v1["task"] == "answer"].copy()
if "LLMJudge" in answer_rows:
    source_ok = answer_rows["CorrectSource"] if "CorrectSource" in answer_rows else pd.Series(True, index=answer_rows.index)
    answer_rows["metric_ok"] = answer_rows["Abstention"].astype(bool) & source_ok.fillna(True).astype(bool)
    answer_rows["judge_ok"] = answer_rows["LLMJudge"].fillna(False).astype(bool)
    disagreements = answer_rows[answer_rows["metric_ok"] != answer_rows["judge_ok"]]
    print(f"Coinciden en {len(answer_rows) - len(disagreements)} de {len(answer_rows)} casos.")
    display(disagreements[["case", "category", "metric_ok", "judge_ok", "LLMJudge_reason"]])
else:
    print("El juez no corrió: revisen JUDGE_MODEL y sus claves.")

In [ ]:
def show_case(case_id: str, rows: dict | None = None) -> None:
    """El caso, lo esperado y la traza del servicio."""
    rows = rows or rows_v1
    case = by_id.get(case_id) or next(c for c in adversarial_cases if c["id"] == case_id)
    print(case_id, "·", case.get("category") or case.get("family", ""))
    print("Esperado:", json.dumps(case.get("expected", case.get("expected_control")), ensure_ascii=False)[:600])
    row = rows[case_id]
    show_record(row["record"]) if row["record"] else print("Rechazado:", row["error"])


CASE_TO_READ = disagreements["case"].iloc[0] if "LLMJudge" in answer_rows and len(disagreements) else TRY_ANSWER
show_case(CASE_TO_READ)

## 7. La puerta

Una **puerta** es la regla que decide si una versión del sistema se puede usar: un mínimo por métrica, escrito antes de mirar los resultados. Pongan los umbrales de los criterios de éxito de su Hito 1. Solo cuentan métricas deterministas: el juez no entra a la puerta.

In [ ]:
GATE = {
    "correct_abstention": 0.8,   # se abstiene en las preguntas sin respuesta
    "with_correct_source": 0.7,  # cita un documento esperado en las que tienen respuesta
    "fields_correct": 0.7,       # campos correctos entre los que usa el servicio
    "adversarial": 1.0,          # casos adversariales detenidos por el control esperado
}


def rates(rows: dict, table: pd.DataFrame) -> dict:
    answers = answer_metrics(cases, answers_of(rows), corpus, "").iloc[0]
    fraction = lambda text: (lambda hits, total: hits / total if total else None)(*map(int, text.split("/")))
    adversarial = table[table["task"] == "adversarial"]
    statuses = [kind for status in statuses_of(rows).values() for kind in status.values() if kind != "not_used"]
    return {
        "correct_abstention": fraction(answers["correct_abstention"]),
        "with_correct_source": fraction(answers["with_correct_source"]),
        "fields_correct": statuses.count("correct") / len(statuses) if statuses else None,
        "adversarial": adversarial["ExpectedControl"].mean() if len(adversarial) else None,
    }


def gate_table(rows: dict, table: pd.DataFrame, label: str) -> pd.DataFrame:
    values = rates(rows, table)
    # Una métrica sin casos (por ejemplo, sin adversariales) queda «sin datos»: ni pasa ni falla.
    return pd.DataFrame([{"version": label, "metric": name, "value": None if values[name] is None else round(values[name], 3),
                          "threshold": threshold, "passes": None if values[name] is None else bool(values[name] >= threshold)}
                         for name, threshold in GATE.items()])


gate_v1 = gate_table(rows_v1, table_v1, "v1")
display(gate_v1)
failing = gate_v1.loc[gate_v1["passes"] == False, "metric"].tolist()
without_data = gate_v1.loc[gate_v1["passes"].isna(), "metric"].tolist()
print("La v1 pasa la puerta." if not failing else "La v1 no pasa la puerta en: " + ", ".join(failing))
if without_data:
    print("Sin datos para:", ", ".join(without_data))

## 8. Análisis de errores

Los casos que fallan algún evaluador determinista, agrupados por la categoría que pusieron en el Hito 1. Para cada grupo, lean dos o tres trazas con `show_case` y anoten la causa probable. Prioricen por el **costo del error** en su dominio, no por la cantidad: una cita falsa sobre un reembolso puede importar más que diez campos omitidos.

In [ ]:
deterministic = [c for c in ("Abstention", "CorrectSource", "ExpectedControl") if c in table_v1]
failed = table_v1[(table_v1[deterministic] == False).any(axis=1) | (table_v1.get("fields_correct", pd.Series(1.0, index=table_v1.index)).fillna(1.0) < 1.0)]
display(failed.groupby(["task", "category"]).size().rename("failed_cases").reset_index())
display(failed[["case", "task", "category"] + [c for c in failed if c.endswith("_reason") and c != "LLMJudge_reason"]])

In [ ]:
show_case(failed["case"].iloc[0] if len(failed) else TRY_EXTRACT)

## 9. Un cambio, medido antes y después

Elijan **un** cambio que nazca de un error que vieron en la sección 8: una instrucción, una regla en `check_answer`, la búsqueda, los datos de una herramienta o el esquema de salida. Un solo cambio, para saber qué causó la diferencia.

1. Abran `src/app.py` en el panel **Archivos** de Colab (doble clic), háganlo y guarden (Ctrl+S). También pueden volver a subir el archivo.
2. Ejecuten la celda siguiente: vuelve a cargar el servicio y corre el mismo `Dataset`.

Un cambio que empeora algo también vale, si lo informan. Después, el `app.py` cambiado va a su repositorio.

In [ ]:
CHANGE = ""  # una frase: qué cambiaron y por qué error

team = load_app(ROOT / "src/app.py")
team_client = service_client(team.app)
ROWS, REPLAY = {}, None
report_v2 = await build_dataset().evaluate(service_task, name="v2", max_concurrency=1)
rows_v2 = dict(ROWS)
save_rows(rows_v2, OUTPUT / "outputs-v2.jsonl")
table_v2 = report_table(report_v2)
table_v2.to_csv(OUTPUT / "report-v2.csv", index=False)
(OUTPUT / "app-v2.py").write_text((ROOT / "src/app.py").read_text(encoding="utf-8"), encoding="utf-8")

display(pd.concat([answer_metrics(cases, answers_of(rows_v1), corpus, "v1"), answer_metrics(cases, answers_of(rows_v2), corpus, "v2")]))
display(pd.concat([field_metrics(cases, statuses_of(rows_v1), "v1"), field_metrics(cases, statuses_of(rows_v2), "v2")]))
display(pd.concat([run_costs(rows_v1, "v1"), run_costs(rows_v2, "v2")]))
display(pd.concat([gate_v1, gate_table(rows_v2, table_v2, "v2")]).pivot(index="metric", columns="version", values="value"))

Los casos que cambiaron de resultado: miren al menos uno que mejoró y, si hay, uno que empeoró.

In [ ]:
columns = [c for c in ("Abstention", "CorrectSource", "fields_correct", "ExpectedControl", "LLMJudge") if c in table_v1 and c in table_v2]
compare = table_v1.set_index("case")[columns].join(table_v2.set_index("case")[columns], lsuffix="_v1", rsuffix="_v2")
text = compare.astype(str)
changed = compare[[any(text.loc[i, f"{c}_v1"] != text.loc[i, f"{c}_v2"] for c in columns) for i in compare.index]]
display(changed)

## 10. Recalcular sin el modelo

El docente va a recalcular sus cifras con las salidas guardadas, sin llamar a ningún modelo ni usar sus claves. La celda hace lo mismo: lee `outputs-v2.jsonl`, corre el `Dataset` sin el juez con una tarea que solo devuelve lo guardado, y compara con lo que vieron arriba. Si algo no coincide, quedó algo fuera de los archivos.

In [ ]:
REPLAY = load_rows(OUTPUT / "outputs-v2.jsonl")
replayed = report_table(await build_dataset(with_judge=False).evaluate(service_task, name="recalculada", progress=False))
REPLAY = None
columns = [c for c in ("Abstention", "CorrectSource", "fields_correct", "ExpectedControl") if c in replayed]
same = replayed.set_index("case")[columns].astype(str).equals(table_v2.set_index("case").loc[replayed["case"], columns].astype(str))
print("Recalculado desde outputs-v2.jsonl:", "coincide." if same else "NO coincide: revisen qué quedó fuera de los archivos.")

## 11. Guardar la evidencia

Completen `DECISIONS` con dos a cuatro oraciones por respuesta, con cifras de sus tablas. La celda guarda `experiment.json` (con `read_answer`, `FIELD_MAP` y la puerta, para recalcular) y arma un ZIP con `labs/lab07-eval/`.

In [ ]:
DECISIONS = {
    # 1. La puerta: qué umbrales eligieron, de qué criterio de éxito del Hito 1 salen, y si la v1 y la v2 la pasan.
    "gate": "",
    # 2. El servicio frente a la línea base del Hito 1: qué mejoró o empeoró, con cifras, y por qué creen que pasó.
    "baseline": "",
    # 3. Las categorías de error que más importan según el costo del error en su dominio, con un caso y su traza.
    "errors": "",
    # 4. Un caso donde el juez y la métrica no coinciden: a cuál le creen y con qué dato.
    "judge": "",
    # 5. El cambio: qué error lo motivó, cifras antes y después en calidad, costo y latencia, y qué empeoró.
    "change": "",
    # 6. Qué cambiarían con más tiempo, y qué falta para recomendar el sistema a su usuario.
    "next": "",
}

import inspect
import shutil
from datetime import datetime, timezone

save_experiment(OUTPUT / "experiment.json", {
    "models": {"service": MODEL, "judge": JUDGE_MODEL}, "schema": schema, "example_data": USE_EXAMPLE,
    "eval_scope": EVAL_SCOPE, "field_map": FIELD_MAP, "extraction_tools": list(EXTRACTION_TOOLS),
    "read_answer": inspect.getsource(read_answer), "abstain_actions": sorted(ABSTAIN_ACTIONS),
    "judge_rubric": judge.rubric, "gate": GATE, "change": CHANGE,
    "gate_v1": gate_v1.to_dict("records"), "gate_v2": gate_table(rows_v2, table_v2, "v2").to_dict("records"),
    "decisions": DECISIONS,
}, config, lab=7)
answers_table.to_csv(OUTPUT / "metrics-answer-v1.csv", index=False)
fields_table.to_csv(OUTPUT / "metrics-fields-v1.csv", index=False)

archive_name = "lab07-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
archive = shutil.make_archive(str(ROOT / archive_name), "zip", root_dir=ROOT, base_dir="labs/lab07-eval")
print("Descarga preparada:", Path(archive).name)
if IN_COLAB:
    from google.colab import files

    files.download(archive)

**Qué va al repositorio del equipo:** la carpeta `labs/lab07-eval/` del ZIP y su `src/app.py` cambiado. Pueden subir los archivos desde el navegador («Add file», «Upload files»). Con la versión que entreguen el 02/12 con la etiqueta `sistema`, el docente vuelve a calcular sus cifras desde `outputs-v1.jsonl` y `outputs-v2.jsonl`.

**Qué va a Canvas:** lo que indique la tarea del laboratorio 7, con el notebook con sus salidas aparte del ZIP. Antes de descargar el notebook, revisen que no tenga una clave pegada en una celda.

## Extensiones opcionales

- **A. La puerta como prueba de regresión.** La misma puerta en `tests/test_gate.py`, para que `pytest` falle si una versión futura baja de los umbrales. Corre sobre las salidas guardadas, sin modelo:

```python
import json
from pathlib import Path

GATE = {"correct_abstention": 0.8}

def test_gate():
    rows = [json.loads(line) for line in Path("labs/lab07-eval/outputs-v2.jsonl").read_text().splitlines()]
    ...  # calculen la métrica como en la sección 7 y comparen con GATE
```

- **B. Un caso adversarial como prueba.** Una prueba que corra su caso adversarial contra el servicio (con el modelo de prueba, como `tests/test_contract.py`) y falle si se quita el control que lo detuvo.

- **C. Repetir para medir la variación.** `dataset.evaluate(..., repeat=3)` corre cada caso tres veces: cuánto cambian las cifras entre corridas del mismo sistema indica qué diferencias entre v1 y v2 son ruido.